In [1]:
!pip install -q torch torchvision tqdm


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from tqdm import tqdm
import os


In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cuda')

In [37]:

DATA_DIR = "/content/classification_data"
BATCH_SIZE = 32
IMG_SIZE = 224
NUM_CLASSES = 25
EPOCHS = 25

In [38]:
import os

for root, dirs, files in os.walk("/content"):
    print(root)
    break

/content


In [19]:
!ls -R /content/classification_data

ls: cannot access '/content/classification_data': No such file or directory


In [21]:
!ls /content

classification_data.zip  sample_data


In [22]:
!unzip classification_data.zip

Streaming output truncated to the last 5000 lines.
  inflating: classification_data/train/couch/couch_38.jpg  
  inflating: classification_data/train/couch/couch_39.jpg  
  inflating: classification_data/train/couch/couch_40.jpg  
  inflating: classification_data/train/couch/couch_42.jpg  
  inflating: classification_data/train/couch/couch_44.jpg  
  inflating: classification_data/train/couch/couch_45.jpg  
  inflating: classification_data/train/couch/couch_46.jpg  
  inflating: classification_data/train/couch/couch_48.jpg  
  inflating: classification_data/train/couch/couch_49.jpg  
  inflating: classification_data/train/couch/couch_5.jpg  
  inflating: classification_data/train/couch/couch_50.jpg  
  inflating: classification_data/train/couch/couch_51.jpg  
  inflating: classification_data/train/couch/couch_52.jpg  
  inflating: classification_data/train/couch/couch_53.jpg  
  inflating: classification_data/train/couch/couch_54.jpg  
  inflating: classification_data/train/couch/couch

In [39]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(0.3, 0.3, 0.3, 0.1),
    transforms.RandomAffine(0, shear=10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [40]:
train_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "train"),
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "val"),
    transform=val_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Train samples:", len(train_dataset))
print("Val samples:", len(val_dataset))

Train samples: 7451
Val samples: 1299


In [41]:
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)

model.classifier = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(model.last_channel, len(train_dataset.classes))
)

model = model.to(device)

In [42]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

scaler = torch.cuda.amp.GradScaler()

/tmp/ipykernel_590/2487718576.py:14: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [43]:
def train_epoch():
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / len(train_loader), correct / total

In [44]:
def validate():
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total

In [45]:
best_val_acc = 0

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch()
    val_acc = validate()

    scheduler.step()

    print(f"Epoch [{epoch+1}/{EPOCHS}] "
          f"Loss: {train_loss:.4f} "
          f"Train Acc: {train_acc:.4f} "
          f"Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "mobilenetv2_best.pth")
        print("Saved new best model")

print("Best Validation Accuracy:", best_val_acc)

/tmp/ipykernel_590/1748641115.py:12: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch [1/25] Loss: 2.5999 Train Acc: 0.2837 Val Acc: 0.4758
Saved new best model
Epoch [2/25] Loss: 2.1491 Train Acc: 0.4436 Val Acc: 0.5235
Saved new best model
Epoch [3/25] Loss: 1.9858 Train Acc: 0.5013 Val Acc: 0.5266
Saved new best model
Epoch [4/25] Loss: 1.8716 Train Acc: 0.5409 Val Acc: 0.5581
Saved new best model
Epoch [5/25] Loss: 1.7637 Train Acc: 0.5817 Val Acc: 0.5735
Saved new best model
Epoch [6/25] Loss: 1.6699 Train Acc: 0.6175 Val Acc: 0.5574
Epoch [7/25] Loss: 1.5887 Train Acc: 0.6516 Val Acc: 0.5727
Epoch [8/25] Loss: 1.5139 Train Acc: 0.6771 Val Acc: 0.5881
Saved new best model
Epoch [9/25] Loss: 1.4335 Train Acc: 0.7029 Val Acc: 0.5928
Saved new best model
Epoch [10/25] Loss: 1.3625 Train Acc: 0.7415 Val Acc: 0.5812
Epoch [11/25] Loss: 1.3101 Train Acc: 0.7614 Val Acc: 0.5781
Epoch [12/25] Loss: 1.2599 Train Acc: 0.7779 Val Acc: 0.5982
Saved new best model
Epoch [13/25] Loss: 1.2223 Train Acc: 0.7999 Val Acc: 0.6043
Saved new best model
Epoch [14/25] Loss: 1.1767 

In [36]:
from google.colab import files
files.download("mobilenetv2_best.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>